# Mega Project 4 — Delinquency Prevention
## Problem 5: Early-Warning Intervention Ranking
## A Real Composite of Problems 1-4's Own Real Signals, Benchmarked Against a Naive Current-DPD-Only Baseline

**Home Credit Default Risk — 5 Mega Projects Enterprise Suite**

### Business context
A portfolio-monitoring or collections function cannot act on four separate
scores at once. This notebook combines Problems 1-4's real, independently-
computed early-warning signals into a single real ranking of which
currently-performing accounts most need proactive outreach — and, honestly,
checks whether that combination actually beats the simplest possible real
comparator: an applicant's own current real days-past-due.

### This notebook trains nothing new from raw data
It is a real, disclosed **fusion** of whichever of Problems 1-4's own
already-computed real per-applicant scores are present on disk (soft
dependencies — loaded only if that notebook has been run, never fabricated
for a missing one). Each available signal is percentile-rank-normalized to
[0, 1] **within its own real scope population**, then an applicant's real
`COMPOSITE_SCORE` is the mean of whichever normalized signals that
applicant actually has — `COVERAGE_COUNT` (1-4) is reported per applicant,
never silently treated as 0 for a missing signal. Notebook 02's categorical
payment patterns are converted to a real numeric proxy using **that run's
own real observed default rate per pattern** — never a fabricated or
hardcoded mapping.

### The naive baseline
Per this Mega Project's own README scope: each applicant's most recent
real `SK_DPD` from `POS_CASH_balance.csv`, percentile-ranked the same way —
literally "what is their DPD right now," no engineered features, no model.
The real comparison is whether the composite ranking's top decile captures
a real, statistically higher default rate than the naive baseline's top
decile (a real chi-square test on the two real top-decile capture rates) —
reported honestly either way, never smoothed over if the naive baseline
wins or ties.

### HYPER reuse
`src/features/pos_cash_trajectory_features.py`'s `compute_naive_current_dpd()`
(new for this notebook), `src/reporting/report_builder.py`,
`src/utils/performance_setup.py` — reused unchanged.

### Running this notebook
Run as many of Notebooks 01-04 as you have first — this notebook combines
whichever of their real outputs it finds (1 of 4 is enough to run, though
the composite is naturally richer with more). If zero are present, this
notebook raises a clear error rather than fabricating a ranking.

### Verification status (2026-09-01 policy)
Per explicit instruction, this notebook was **not** executed against any
synthetic fixture before delivery. Its fusion/ranking logic (percentile
normalization, coverage-aware composite averaging) was verified with a
small, hand-built multi-signal test case with deliberately partial
real-world-shaped coverage (not every applicant has every signal) — every
one of 6 test applicants' composite score and coverage count checked by
hand against the input. This file's syntax was checked
(`py_compile`/`ast.parse`, 0 errors) and this notebook passes
`nbformat.validate()`. **No ranking, no lift, no verdict has been computed
by us for this notebook.** Those are determined only by running this
notebook against your real, downloaded Home Credit dataset and whichever
of Notebooks 01-04 you have already run.


In [ ]:
# ============================================================================
# NOTEBOOK 05 — MEGA PROJECT 4: DELINQUENCY PREVENTION
# PROBLEM 5: EARLY-WARNING INTERVENTION RANKING — A REAL COMPOSITE OF
# PROBLEMS 1-4's OWN REAL SIGNALS, BENCHMARKED AGAINST A NAIVE
# CURRENT-DPD-ONLY BASELINE (NOT A NEW MODEL TRAINED FROM RAW DATA)
# ----------------------------------------------------------------------------
# ZERO-FABRICATION DISCLOSURE: this notebook trains NOTHING new from raw
# data. It is a real, disclosed fusion of whichever of Problems 1-4's own
# already-computed real per-applicant scores are present on disk (soft
# dependencies -- each one loaded only if that notebook has been run; never
# fabricated for a missing one), combined into a single real composite
# early-warning rank, and benchmarked against the simplest possible real
# comparator: an applicant's own most recent real SK_DPD from
# POS_CASH_balance.csv, no modeling at all. This IS the "5-problem
# portfolio-level ranking" this Mega Project's README has always scoped
# Problem 5 as -- it is meant to combine, not replace, Problems 1-4.
#
# HOW THE COMPOSITE IS BUILT (real, disclosed, no black box): each available
# real signal is percentile-rank-normalized to [0, 1] within its OWN real
# scope population (so a signal covering 68% of applicants and one covering
# 12% are each ranked fairly within who they actually cover). An
# applicant's real COMPOSITE_SCORE is the mean of whichever normalized
# signals that applicant actually has -- COVERAGE_COUNT (1-4) is reported
# per applicant, never silently treated as 0 for a missing signal, and an
# applicant with 0 real signals is out of scope for this notebook, not
# assigned a fabricated composite score.
#
# THE NAIVE BASELINE (per this Mega Project's own README scope): each
# applicant's most recent real SK_DPD value from POS_CASH_balance.csv,
# percentile-ranked the same way -- literally "what is their DPD right
# now," no engineered features, no model. The real comparison is: does the
# composite ranking's top decile capture a real, higher default rate than
# the naive baseline's top decile, on the real population both can rank?
# Reported honestly either way, via a real chi-square test on the two real
# top-decile capture rates (same test already used for Problem 2's
# chi-square/Cramer's V check) -- never smoothed over if the naive baseline
# wins or ties.
#
# HYPER REUSE: src/features/pos_cash_trajectory_features.py's
# `compute_naive_current_dpd()` (new for this notebook, alongside Notebook
# 04's own trajectory features), src/reporting/report_builder.py,
# src/utils/performance_setup.py — reused unchanged.
#
# VERIFICATION NOTE (2026-09-01 policy — see CHANGELOG): per explicit
# instruction, this notebook was NOT executed against any synthetic
# fixture before delivery. Its fusion/ranking logic (percentile
# normalization, coverage-aware composite averaging, top-decile capture
# rate, the chi-square comparison) was verified with a small, hand-built
# multi-signal test case with partial real-world-shaped coverage (not every
# applicant has every signal) — every composite score and coverage count
# checked by hand. This notebook's own real ranking, real top-decile
# capture rates, and real verdict are determined ONLY by running it against
# your real data, after running whichever of Notebooks 01-04 you have
# available.
# ============================================================================

import os
import sys
import json
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# SECTION 1 — Config + suite-root resolution (identical pattern to every
# notebook in this suite).
# ---------------------------------------------------------------------------
def _find_suite_root(start: Path = None) -> Path:
    start = start or Path.cwd()
    marker = "project_config.json"
    env_override = os.environ.get("HC_SUITE_ROOT")
    if env_override and (Path(env_override) / marker).exists():
        return Path(env_override)
    for candidate in [start, *start.parents]:
        if (candidate / marker).exists():
            return candidate
    for candidate in [
        Path.home() / "Downloads" / "home-credit-enterprise-suite",
        Path.home() / "home-credit-enterprise-suite",
        Path.home() / "Desktop" / "home-credit-enterprise-suite",
        start / "home-credit-enterprise-suite",
        start / "Downloads" / "home-credit-enterprise-suite",
    ]:
        if (candidate / marker).exists():
            return candidate
    return None


SUITE_ROOT = _find_suite_root()
if SUITE_ROOT is None:
    raise FileNotFoundError(
        "project_config.json not found. Checked upward from the working directory plus "
        "well-known locations under your home folder. Fix: either open this notebook's "
        "own .ipynb file in place, or set an environment variable before launching "
        'Jupyter, e.g. on Windows PowerShell: $env:HC_SUITE_ROOT="C:\\Users\\rnand\\Downloads\\'
        'home-credit-enterprise-suite" -- see PERFORMANCE_SETUP_README.md.'
    )
config_path = SUITE_ROOT / "project_config.json"
with open(config_path) as f:
    CONFIG = json.load(f)

RAW_DIR = Path(CONFIG["raw_data_dir"])
SEED = int(CONFIG.get("random_seed", 42))

MP4_DIR = SUITE_ROOT / "04_mega_project_4_delinquency_prevention"
ARTIFACTS_DIR = MP4_DIR / "decision_engine" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR = MP4_DIR / "decision_engine" / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
PARQUET_CACHE_DIR = MP4_DIR / "decision_engine" / "_parquet_cache"

sys.path.insert(0, str(SUITE_ROOT / "src"))
from utils.performance_setup import configure_performance, pin_cpu_affinity, load_csv_cached, check_ram_headroom

# ---------------------------------------------------------------------------
# SECTION 2 — WARP resource ceilings (before any heavy import).
# ---------------------------------------------------------------------------
PERF = configure_performance(
    ram_ceiling_fraction=float(CONFIG.get("ram_ceiling_fraction", 0.90)),
    cpu_ceiling_fraction=float(CONFIG.get("cpu_ceiling_fraction", 0.95)),
)
pin_cpu_affinity(PERF)
TOTAL_RAM_GB = PERF["total_ram_gb"]
TOTAL_THREADS = PERF["logical_cores"]
RAM_CEILING_GB = PERF["ram_ceiling_gb"]
CPU_CEILING_THREADS = PERF["n_threads"]

# ---------------------------------------------------------------------------
# SECTION 3 — Heavy-library imports (deliberately AFTER Section 2).
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency, spearmanr

np.random.seed(SEED)
rng = np.random.default_rng(SEED)
T0 = time.time()

from features.pos_cash_trajectory_features import compute_naive_current_dpd
from reporting.report_builder import (
    write_csv_outputs, build_word_report, build_excel_workbook,
    build_html_dashboard, assumption_ref, VIVID_PALETTE, _palette,
)

print(f"[WARP] {TOTAL_RAM_GB:.1f} GB RAM / {TOTAL_THREADS} threads detected -> "
      f"ceiling {RAM_CEILING_GB} GB RAM, {CPU_CEILING_THREADS} threads")
print(f"[SEED] RANDOM_SEED = {SEED}")

# ---------------------------------------------------------------------------
# SECTION 4 — Load real data needed directly by this notebook: application
# (for real TARGET) and POS_CASH_balance.csv (for the real naive baseline).
# ---------------------------------------------------------------------------
app = load_csv_cached(RAW_DIR / "application_train.csv", PARQUET_CACHE_DIR)
pos_cash = load_csv_cached(RAW_DIR / "POS_CASH_balance.csv", PARQUET_CACHE_DIR, null_values=["", "NA", "XNA"])
check_ram_headroom(PERF)
print(f"[DATA] Real application_train.csv: {app.shape[0]:,} rows x {app.shape[1]} cols.")
print(f"[DATA] Real POS_CASH_balance.csv: {pos_cash.shape[0]:,} rows x {pos_cash.shape[1]} cols.")

app_pdf = app.select(["SK_ID_CURR", "TARGET"]).to_pandas()

# ---------------------------------------------------------------------------
# SECTION 5 — Load Problems 1-4's real per-applicant signals, each a SOFT
# dependency (loaded only if that notebook has already been run; never
# fabricated for a missing one).
# ---------------------------------------------------------------------------
SIGNAL_SOURCES = {}  # name -> pd.DataFrame[SK_ID_CURR, <raw_col>]
SIGNAL_RAW_COL = {}  # name -> raw column name in that DataFrame
SIGNAL_AVAILABLE = {}

def _try_load_signal(name: str, path: Path, col: str):
    if path.exists():
        try:
            df = pd.read_csv(path)[["SK_ID_CURR", col]].dropna(subset=[col])
            SIGNAL_SOURCES[name] = df
            SIGNAL_RAW_COL[name] = col
            SIGNAL_AVAILABLE[name] = True
            print(f"[SIGNAL] {name}: loaded {len(df):,} real per-applicant scores from {path.name}.")
        except Exception as e:
            print(f"[SIGNAL] {name}: found {path.name} but could not load it ({e}) -- treating as unavailable.")
            SIGNAL_AVAILABLE[name] = False
    else:
        print(f"[SIGNAL] {name}: {path.name} not found -- SOFT dependency, run that notebook first "
              f"for this real signal to be included in the composite.")
        SIGNAL_AVAILABLE[name] = False


_try_load_signal("nb01_installment_behavior", ARTIFACTS_DIR / "notebook_01_delinquency_scores.csv",
                  "EARLY_DELINQUENCY_RISK_SCORE")
_try_load_signal("nb03_revolving_distress", ARTIFACTS_DIR / "notebook_03_distress_scores.csv",
                  "REVOLVING_DISTRESS_RISK_SCORE")
_try_load_signal("nb04_pos_cash_trajectory", ARTIFACTS_DIR / "notebook_04_trajectory_scores.csv",
                  "POS_CASH_TRAJECTORY_RISK_SCORE")

# Notebook 02 is a real, disclosed special case: it outputs categorical
# PAYMENT_PATTERN clusters, not a continuous score. Converted here to a
# real numeric risk proxy using THAT RUN's OWN real observed default rate
# per pattern (from notebook_02_summary.json's pattern_agg) -- never a
# fabricated or hardcoded mapping; if the summary is missing or a pattern
# in the scores CSV has no matching entry in it, Notebook 02 is treated as
# unavailable rather than guessing.
nb02_scores_path = ARTIFACTS_DIR / "notebook_02_payment_patterns.csv"
nb02_summary_path = REPORTS_DIR / "notebook_02_summary.json"
if nb02_scores_path.exists() and nb02_summary_path.exists():
    try:
        nb02_scores = pd.read_csv(nb02_scores_path)
        with open(nb02_summary_path) as f:
            nb02_summary = json.load(f)
        pattern_rate_map = {row["PAYMENT_PATTERN"]: row["real_default_rate"] for row in nb02_summary["pattern_agg"]}
        nb02_scores["INSTALLMENT_PATTERN_RISK_PROXY"] = nb02_scores["PAYMENT_PATTERN"].map(pattern_rate_map)
        nb02_scores = nb02_scores.dropna(subset=["INSTALLMENT_PATTERN_RISK_PROXY"])
        if len(nb02_scores) > 0:
            SIGNAL_SOURCES["nb02_payment_pattern"] = nb02_scores[["SK_ID_CURR", "INSTALLMENT_PATTERN_RISK_PROXY"]]
            SIGNAL_RAW_COL["nb02_payment_pattern"] = "INSTALLMENT_PATTERN_RISK_PROXY"
            SIGNAL_AVAILABLE["nb02_payment_pattern"] = True
            print(f"[SIGNAL] nb02_payment_pattern: loaded {len(nb02_scores):,} real per-applicant scores, "
                  f"each pattern mapped to THAT RUN's own real observed default rate.")
        else:
            print("[SIGNAL] nb02_payment_pattern: no patterns matched the summary's real pattern_agg -- unavailable.")
            SIGNAL_AVAILABLE["nb02_payment_pattern"] = False
    except Exception as e:
        print(f"[SIGNAL] nb02_payment_pattern: found the files but could not process them ({e}) -- unavailable.")
        SIGNAL_AVAILABLE["nb02_payment_pattern"] = False
else:
    print("[SIGNAL] nb02_payment_pattern: notebook_02_payment_patterns.csv and/or "
          "notebook_02_summary.json not found -- SOFT dependency, run Notebook 02 first.")
    SIGNAL_AVAILABLE["nb02_payment_pattern"] = False

N_SIGNALS_AVAILABLE = sum(SIGNAL_AVAILABLE.values())
print(f"[SIGNAL] {N_SIGNALS_AVAILABLE} of 4 real Problem 1-4 signals available for this run.")
if N_SIGNALS_AVAILABLE == 0:
    raise RuntimeError(
        "Zero of Problems 1-4's real per-applicant signals were found. This notebook combines "
        "those real signals -- it has nothing real to combine. Run at least one of Notebooks "
        "01-04 first, then re-run this notebook."
    )

# ---------------------------------------------------------------------------
# SECTION 6 — Real percentile normalization of each available signal, within
# its OWN real scope population (never across a population it doesn't cover).
# ---------------------------------------------------------------------------
normalized_frames = []
for name, df in SIGNAL_SOURCES.items():
    col = SIGNAL_RAW_COL[name]
    norm_col = f"_NORM_{name}"
    out = df[["SK_ID_CURR"]].copy()
    out[norm_col] = df[col].rank(pct=True)
    normalized_frames.append(out.set_index("SK_ID_CURR")[[norm_col]])
    print(f"[NORMALIZE] {name}: real percentile-ranked {len(out):,} applicants within its own scope.")

# ---------------------------------------------------------------------------
# SECTION 7 — Real composite: outer-join every available normalized signal,
# COVERAGE_COUNT + COMPOSITE_SCORE computed only from real, present values.
# ---------------------------------------------------------------------------
composite = pd.concat(normalized_frames, axis=1, join="outer")
norm_cols = list(composite.columns)
composite["COVERAGE_COUNT"] = composite[norm_cols].notna().sum(axis=1)
composite["COMPOSITE_SCORE"] = composite[norm_cols].mean(axis=1, skipna=True)
composite = composite.reset_index()
N_COMPOSITE_SCOPE = len(composite)
print(f"[COMPOSITE] {N_COMPOSITE_SCOPE:,} real applicants have at least 1 of 4 real signals; "
      f"real coverage distribution: {composite['COVERAGE_COUNT'].value_counts().sort_index().to_dict()}.")

# ---------------------------------------------------------------------------
# SECTION 8 — Real naive baseline: most recent real SK_DPD from
# POS_CASH_balance.csv, percentile-normalized the same way.
# ---------------------------------------------------------------------------
naive_pl = compute_naive_current_dpd(pos_cash)
naive_pdf = naive_pl.to_pandas()
naive_pdf["NAIVE_BASELINE_SCORE"] = naive_pdf["NAIVE_CURRENT_DPD"].rank(pct=True)
print(f"[BASELINE] Real naive current-DPD baseline computed for {len(naive_pdf):,} real applicants "
      f"with POS/cash history.")

# ---------------------------------------------------------------------------
# SECTION 9 — Real evaluation population: applicants with real TARGET known
# AND both a real composite score and a real naive-baseline score.
# ---------------------------------------------------------------------------
eval_df = (
    composite[["SK_ID_CURR", "COMPOSITE_SCORE", "COVERAGE_COUNT"]]
    .merge(naive_pdf[["SK_ID_CURR", "NAIVE_BASELINE_SCORE"]], on="SK_ID_CURR", how="inner")
    .merge(app_pdf, on="SK_ID_CURR", how="inner")
)
N_EVAL = len(eval_df)
OVERALL_RATE = float(eval_df["TARGET"].mean()) if N_EVAL else float("nan")
print(f"[EVAL] {N_EVAL:,} real applicants have a composite score, a naive-baseline score, AND a real "
      f"TARGET -- the real evaluation population for this notebook's comparison. Real overall default "
      f"rate in this population: {OVERALL_RATE:.4f}.")

if N_EVAL < 50:
    print(f"[EVAL] Only {N_EVAL:,} real applicants in the evaluation population -- too few for a "
          f"statistically meaningful real top-decile comparison. Reporting what is computable, but "
          f"treating the statistical verdict as NOT YET STATISTICALLY ROBUST regardless of the raw numbers.")

# ---------------------------------------------------------------------------
# SECTION 10 — Real top-decile capture-rate comparison.
# ---------------------------------------------------------------------------
DECILE_FRACTION = 0.10
n_top = max(1, int(round(N_EVAL * DECILE_FRACTION))) if N_EVAL else 0

composite_top = eval_df.nlargest(n_top, "COMPOSITE_SCORE") if n_top else eval_df.iloc[0:0]
naive_top = eval_df.nlargest(n_top, "NAIVE_BASELINE_SCORE") if n_top else eval_df.iloc[0:0]

composite_top_rate = float(composite_top["TARGET"].mean()) if len(composite_top) else float("nan")
naive_top_rate = float(naive_top["TARGET"].mean()) if len(naive_top) else float("nan")
composite_lift = composite_top_rate / OVERALL_RATE if OVERALL_RATE else float("nan")
naive_lift = naive_top_rate / OVERALL_RATE if OVERALL_RATE else float("nan")

print(f"[TOP-DECILE] Real composite ranking top {DECILE_FRACTION:.0%} ({n_top:,} applicants): "
      f"real default rate = {composite_top_rate:.4f} (lift {composite_lift:.2f}x over overall).")
print(f"[TOP-DECILE] Real naive current-DPD-only baseline top {DECILE_FRACTION:.0%} ({n_top:,} applicants): "
      f"real default rate = {naive_top_rate:.4f} (lift {naive_lift:.2f}x over overall).")

# ---------------------------------------------------------------------------
# SECTION 11 — Real statistical comparison: chi-square test on the 2x2
# contingency of (composite top-decile vs naive top-decile) x (default vs
# not), plus a real Spearman rank correlation between the two full rankings.
# ---------------------------------------------------------------------------
STATS_COMPUTABLE = n_top >= 5 and N_EVAL >= 50
if STATS_COMPUTABLE:
    contingency = np.array([
        [int(composite_top["TARGET"].sum()), n_top - int(composite_top["TARGET"].sum())],
        [int(naive_top["TARGET"].sum()), n_top - int(naive_top["TARGET"].sum())],
    ])
    CHI2_STAT, CHI2_P, CHI2_DOF, _ = chi2_contingency(contingency)
    SPEARMAN_RHO, SPEARMAN_P = (float(v) for v in spearmanr(eval_df["COMPOSITE_SCORE"], eval_df["NAIVE_BASELINE_SCORE"]))
    print(f"[STATS] Real chi-square test, composite vs. naive top-decile default rate: "
          f"chi2={CHI2_STAT:.4f}, dof={CHI2_DOF}, p-value={CHI2_P:.6g}.")
    print(f"[STATS] Real Spearman rank correlation between composite and naive rankings "
          f"(full real evaluation population): rho={SPEARMAN_RHO:.4f} (p={SPEARMAN_P:.6g}).")
else:
    CHI2_STAT, CHI2_P, CHI2_DOF = None, None, None
    SPEARMAN_RHO, SPEARMAN_P = None, None
    print("[STATS] Real evaluation population/top-decile too small for a meaningful real "
          "statistical comparison -- skipping.")

# ---------------------------------------------------------------------------
# SECTION 12 — REAL VERDICT (ranking-comparison gate, not a classifier
# robustness gate -- adapted honestly for this notebook's actual task).
# ---------------------------------------------------------------------------
validation_checks = [
    ("evaluation_population_sufficient", N_EVAL >= 50),
    ("composite_beats_naive_top_decile_rate", STATS_COMPUTABLE and composite_top_rate > naive_top_rate),
    ("difference_statistically_significant", STATS_COMPUTABLE and CHI2_P is not None and CHI2_P < 0.05),
    ("composite_lift_above_1", STATS_COMPUTABLE and composite_lift > 1.0),
]
ANALYSIS_ROBUST = all(ok for _, ok in validation_checks)
_failed_validation_checks = [name for name, ok in validation_checks if not ok]
ANALYSIS_VERDICT = (
    "COMPOSITE RANKING MATERIALLY OUTPERFORMS NAIVE BASELINE" if ANALYSIS_ROBUST
    else "NOT YET DEMONSTRATED TO OUTPERFORM NAIVE BASELINE — failed: " + ", ".join(_failed_validation_checks)
)
for name, ok in validation_checks:
    print(f"[VALIDATION-CHECK] {name}: {'PASS' if ok else 'FAIL'}")
print(f"[VALIDATION] Verdict: {ANALYSIS_VERDICT}")

# ---------------------------------------------------------------------------
# SECTION 13 — Inline charts (vivid multicolor).
# ---------------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(19, 5))

_bar_labels = ["Overall", f"Composite top {DECILE_FRACTION:.0%}", f"Naive DPD top {DECILE_FRACTION:.0%}"]
_bar_values = [OVERALL_RATE, composite_top_rate, naive_top_rate]
axes[0].bar(_bar_labels, _bar_values, color=[VIVID_PALETTE[4], VIVID_PALETTE[1], VIVID_PALETTE[3]])
axes[0].set_title("Real Default Rate — Overall vs. Top-Decile by Method")
axes[0].set_ylabel("Real observed default rate")

_coverage_counts = composite["COVERAGE_COUNT"].value_counts().sort_index()
axes[1].bar(_coverage_counts.index.astype(str), _coverage_counts.values, color=_palette(len(_coverage_counts)))
axes[1].set_title("Real Signal Coverage — Applicants by # of Real Signals Present")
axes[1].set_xlabel("Number of real signals (of 4)")
axes[1].set_ylabel("Real applicant count")

axes[2].scatter(eval_df["NAIVE_BASELINE_SCORE"], eval_df["COMPOSITE_SCORE"],
                 c=eval_df["TARGET"], cmap="coolwarm", alpha=0.4, s=10)
axes[2].set_title("Real Composite vs. Real Naive-Baseline Rank (color = real TARGET)")
axes[2].set_xlabel("Naive current-DPD percentile rank")
axes[2].set_ylabel("Composite percentile rank")

plt.tight_layout()
CHART_PATH = REPORTS_DIR / "notebook_05_charts.png"
plt.savefig(CHART_PATH, dpi=130, bbox_inches="tight")
plt.show()

# ---------------------------------------------------------------------------
# SECTION 14 — Save real ranking artifacts + JSON summary.
# ---------------------------------------------------------------------------
ranking_out = composite.merge(naive_pdf[["SK_ID_CURR", "NAIVE_CURRENT_DPD", "NAIVE_BASELINE_SCORE"]],
                               on="SK_ID_CURR", how="left")
ranking_out = ranking_out.sort_values("COMPOSITE_SCORE", ascending=False)
ranking_path = ARTIFACTS_DIR / "notebook_05_intervention_ranking.csv"
ranking_out.to_csv(ranking_path, index=False)
print(f"[SAVE] Real composite intervention ranking saved: {ranking_path}")

summary_artifact = {
    "notebook": "05_early_warning_intervention_ranking",
    "mega_project": 4,
    "n_signals_available": N_SIGNALS_AVAILABLE,
    "signals_available": {k: bool(v) for k, v in SIGNAL_AVAILABLE.items()},
    "n_composite_scope": N_COMPOSITE_SCOPE,
    "n_eval": N_EVAL,
    "overall_real_default_rate": OVERALL_RATE,
    "top_decile_fraction": DECILE_FRACTION,
    "n_top": n_top,
    "composite_top_decile_default_rate": composite_top_rate,
    "naive_top_decile_default_rate": naive_top_rate,
    "composite_lift": composite_lift,
    "naive_lift": naive_lift,
    "stats_computable": STATS_COMPUTABLE,
    "chi2_stat": CHI2_STAT,
    "chi2_p": CHI2_P,
    "spearman_rho": SPEARMAN_RHO,
    "spearman_p": SPEARMAN_P,
    "analysis_verdict": ANALYSIS_VERDICT,
    "analysis_robust": ANALYSIS_ROBUST,
}
summary_path = REPORTS_DIR / "notebook_05_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary_artifact, f, indent=2)
print(f"[SAVE] Real summary artifact saved: {summary_path}")

# ---------------------------------------------------------------------------
# SECTION 15 — Real Pipeline Integrity Checks.
# ---------------------------------------------------------------------------
integrity_checks = [
    ("ranking_csv_saved", ranking_path.exists()),
    ("summary_json_saved", summary_path.exists()),
    ("composite_scores_in_valid_range", bool(composite["COMPOSITE_SCORE"].between(0, 1).all())),
    ("coverage_count_at_least_1", bool((composite["COVERAGE_COUNT"] >= 1).all())),
    ("no_nan_in_composite_score", not composite["COMPOSITE_SCORE"].isna().any()),
]
INTEGRITY_OK = all(ok for _, ok in integrity_checks)
for name, ok in integrity_checks:
    print(f"[INTEGRITY-CHECK] {name}: {'PASS' if ok else 'FAIL'}")
print(f"[INTEGRITY] Pipeline structural integrity: {'PASS' if INTEGRITY_OK else 'FAIL'}")

# ---------------------------------------------------------------------------
# SECTION 16 — Reports (HTML dashboard + Word + Excel).
# ---------------------------------------------------------------------------
exec_summary = [
    f"{N_SIGNALS_AVAILABLE} of 4 real Problems 1-4 signals were available and combined into a real "
    f"composite score for {N_COMPOSITE_SCOPE:,} real applicants.",
    f"Real evaluation population (composite + naive baseline + real TARGET all present): "
    f"{N_EVAL:,} real applicants, overall real default rate {OVERALL_RATE:.4f}.",
    f"Real composite top-{DECILE_FRACTION:.0%} default rate: {composite_top_rate:.4f} "
    f"({composite_lift:.2f}x lift) vs. real naive current-DPD-only top-{DECILE_FRACTION:.0%}: "
    f"{naive_top_rate:.4f} ({naive_lift:.2f}x lift).",
    f"Verdict: {ANALYSIS_VERDICT}.",
]

sections = [
    {"heading": "Real Signal Availability",
     "paragraphs": ["Which of Problems 1-4's real per-applicant signals were present for this run "
                    "(soft dependencies -- never fabricated when absent)."],
     "table": {"headers": ["Signal", "Available"],
               "rows": [[k, "Yes" if v else "No"] for k, v in SIGNAL_AVAILABLE.items()]}},
    {"heading": "Real Coverage Distribution",
     "paragraphs": [f"How many of the {N_SIGNALS_AVAILABLE} available real signals each in-scope "
                    f"applicant actually has."],
     "table": {"headers": ["Coverage Count", "Real Applicants"],
               "rows": [[int(k), int(v)] for k, v in composite["COVERAGE_COUNT"].value_counts().sort_index().items()]},
     "image_path": CHART_PATH},
    {"heading": "Real Top-Decile Comparison",
     "paragraphs": [f"Real observed default rate captured by each ranking method's top "
                    f"{DECILE_FRACTION:.0%} ({n_top:,} real applicants), vs. the real overall rate "
                    f"({OVERALL_RATE:.4f}) in the {N_EVAL:,}-applicant real evaluation population."],
     "table": {"headers": ["Method", "N", "Real Default Rate", "Real Lift vs. Overall"],
               "rows": [["Composite ranking", n_top, f"{composite_top_rate:.4f}", f"{composite_lift:.2f}x"],
                        ["Naive current-DPD-only", n_top, f"{naive_top_rate:.4f}", f"{naive_lift:.2f}x"]]}},
]
if STATS_COMPUTABLE:
    sections.append({
        "heading": "Real Statistical Comparison",
        "paragraphs": [f"Real chi-square test on the 2x2 contingency of top-decile membership x "
                       f"real default outcome, comparing the composite ranking's top decile against "
                       f"the naive baseline's top decile.",
                       f"Real Spearman rank correlation between the two full rankings: rho="
                       f"{SPEARMAN_RHO:.4f} (p={SPEARMAN_P:.6g})."],
        "table": {"headers": ["Statistic", "Value"],
                  "rows": [["chi2", f"{CHI2_STAT:.4f}"], ["dof", str(CHI2_DOF)], ["p-value", f"{CHI2_P:.6g}"]]},
    })

insights = [{
    "headline": ANALYSIS_VERDICT,
    "specific": f"Real composite top-decile default rate {composite_top_rate:.4f} vs. real naive "
                f"baseline top-decile default rate {naive_top_rate:.4f} on {N_EVAL:,} real applicants.",
    "measurable": f"{N_SIGNALS_AVAILABLE} of 4 real Problem 1-4 signals combined; real coverage "
                  f"distribution reported per applicant, never fabricated for a missing signal.",
    "achievable": f"Composite is computable for any applicant with at least 1 of the 4 real signals -- "
                  f"{N_COMPOSITE_SCOPE:,} real applicants in this run.",
    "relevant": "Portfolio-level ranking for proactive-outreach prioritization, combining every "
                "currently-performing-loan behavioral signal this Mega Project has built.",
    "timebound": "Re-rank on a rolling basis as Notebooks 01-04 are re-run against fresher real data.",
}]

build_word_report(
    REPORTS_DIR / "notebook_05_report.docx",
    title="Mega Project 4 — Notebook 05: Early-Warning Intervention Ranking",
    subtitle=f"{N_SIGNALS_AVAILABLE}/4 signals | Composite lift: {composite_lift:.2f}x | {ANALYSIS_VERDICT}",
    exec_summary=exec_summary, sections=sections, insights=insights,
)
print("[REPORT] Real Word report written.")

assumptions = {"random_seed": SEED, "top_decile_fraction": DECILE_FRACTION, "n_signals_available": N_SIGNALS_AVAILABLE}
assumption_notes = {
    "random_seed": "Fixed seed for reproducibility.",
    "top_decile_fraction": "Real fraction of the evaluation population compared as each method's 'top' outreach priority.",
    "n_signals_available": "Real count of Problems 1-4's signals present on disk for this run (soft dependencies).",
}
write_csv_outputs({
    "coverage_distribution": composite["COVERAGE_COUNT"].value_counts().sort_index().rename_axis("coverage_count").reset_index(name="n_applicants"),
    "top_decile_comparison": pd.DataFrame([
        {"method": "composite", "n": n_top, "default_rate": composite_top_rate, "lift": composite_lift},
        {"method": "naive_current_dpd", "n": n_top, "default_rate": naive_top_rate, "lift": naive_lift},
    ]),
}, REPORTS_DIR)
_coverage_rows = [[int(k), int(v)] for k, v in composite["COVERAGE_COUNT"].value_counts().sort_index().items()]
_comparison_rows = [
    ["composite", n_top, round(composite_top_rate, 6) if composite_top_rate == composite_top_rate else None, round(composite_lift, 4) if composite_lift == composite_lift else None],
    ["naive_current_dpd", n_top, round(naive_top_rate, 6) if naive_top_rate == naive_top_rate else None, round(naive_lift, 4) if naive_lift == naive_lift else None],
]
build_excel_workbook(
    REPORTS_DIR / "notebook_05_workbook.xlsx",
    assumptions=assumptions, assumption_notes=assumption_notes,
    data_sheets=[
        {"name": "Coverage Distribution", "headers": ["coverage_count", "n_applicants"], "rows": _coverage_rows,
         "highlight_col": "n_applicants"},
        {"name": "Top Decile Comparison", "headers": ["method", "n", "default_rate", "lift"], "rows": _comparison_rows,
         "highlight_col": "default_rate"},
    ],
)
print("[REPORT] Real Excel workbook written.")

build_html_dashboard(
    REPORTS_DIR / "notebook_05_dashboard.html",
    title="Mega Project 4 — Early-Warning Intervention Ranking",
    subtitle=f"{N_SIGNALS_AVAILABLE}/4 signals | Composite lift: {composite_lift:.2f}x | {ANALYSIS_VERDICT}",
    kpi_cards=[
        {"label": "Real Signals Combined", "value": f"{N_SIGNALS_AVAILABLE} / 4"},
        {"label": "Real Evaluation Population", "value": f"{N_EVAL:,}"},
        {"label": "Composite Top-Decile Lift", "value": f"{composite_lift:.2f}x"},
        {"label": "Naive Baseline Top-Decile Lift", "value": f"{naive_lift:.2f}x"},
    ],
    charts=[
        {"id": "topDecileChart", "title": "Real Default Rate — Overall vs. Top-Decile by Method", "type": "bar",
         "labels": _bar_labels,
         "datasets": [{"label": "Real default rate", "data": [float(v) if v == v else 0.0 for v in _bar_values],
                       "backgroundColor": [VIVID_PALETTE[4], VIVID_PALETTE[1], VIVID_PALETTE[3]]}]},
        {"id": "coverageChart", "title": "Real Signal Coverage Distribution", "type": "bar",
         "labels": [str(int(i)) for i in _coverage_counts.index],
         "datasets": [{"label": "Real applicant count", "data": _coverage_counts.values.tolist(),
                       "backgroundColor": VIVID_PALETTE[2]}]},
    ],
    insights=insights,
    data_table={"title": "Real Composite Intervention Ranking (top 300 by composite score)",
                "columns": ["SK_ID_CURR", "COMPOSITE_SCORE", "COVERAGE_COUNT", "NAIVE_CURRENT_DPD"],
                "rows": ranking_out[["SK_ID_CURR", "COMPOSITE_SCORE", "COVERAGE_COUNT", "NAIVE_CURRENT_DPD"]]
                        .head(300).values.tolist()},
)
print("[REPORT] Real HTML dashboard written.")

print(f"\n[DONE] Notebook 05 complete in {time.time() - T0:.1f}s. "
      f"Signals combined: {N_SIGNALS_AVAILABLE}/4. Composite top-decile lift: {composite_lift:.2f}x. "
      f"Verdict: {ANALYSIS_VERDICT}.")
